In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("data/education.db")

tertiary_male = pd.read_csv(
    "data/raw/enrollment/TERTIARY Male School Enrollment % DATA.csv.csv",
    skiprows=4,
    engine="python"
)

tertiary_female = pd.read_csv(
    "data/raw/enrollment/TERTIARY Female School Enrollment % DATA (3).csv",
    skiprows=4,
    engine="python"
)

In [2]:
tertiary_male.to_sql("tertiary_male_raw", conn, if_exists="replace", index=False)
tertiary_female.to_sql("tertiary_female_raw", conn, if_exists="replace", index=False)

266

In [3]:
pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE name LIKE 'tertiary%';
""", conn)

,name
0,tertiary_male_raw
1,tertiary_female_raw


In [4]:
conn.executescript("""
DROP TABLE IF EXISTS tertiary_male_clean;
DROP TABLE IF EXISTS tertiary_female_clean;

CREATE TABLE tertiary_male_clean AS
SELECT *
FROM tertiary_male_raw
WHERE "Country Code" IN (
    'AFE','AFW','ARB','AUS','EAS','EUU','LCN','NAC','SAS',
    'LIC','LMC','UMC','HIC'
);

CREATE TABLE tertiary_female_clean AS
SELECT *
FROM tertiary_female_raw
WHERE "Country Code" IN (
    'AFE','AFW','ARB','AUS','EAS','EUU','LCN','NAC','SAS',
    'LIC','LMC','UMC','HIC'
);
""")

In [5]:
conn.executescript("""
DROP TABLE IF EXISTS tertiary_gender_merged;

CREATE TABLE tertiary_gender_merged AS
SELECT
    m."Country Name" AS country_name,
    m."Country Code" AS country_code,
    m."2000" AS male_2000,
    f."2000" AS female_2000
FROM tertiary_male_clean m
JOIN tertiary_female_clean f
    ON m."Country Code" = f."Country Code";
""")

In [6]:
pd.read_sql("SELECT COUNT(*) FROM tertiary_gender_merged;", conn)

,COUNT(*)
0,13


In [7]:
pd.read_sql("SELECT * FROM tertiary_male_clean", conn)\
  .to_csv("data/cleaned/tertiary_male_clean.csv", index=False)

pd.read_sql("SELECT * FROM tertiary_female_clean", conn)\
  .to_csv("data/cleaned/tertiary_female_clean.csv", index=False)

pd.read_sql("SELECT * FROM tertiary_gender_merged", conn)\
  .to_csv("data/cleaned/tertiary_gender_merged.csv", index=False)